In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.cluster import DBSCAN
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from IPython.display import display # Για καλύτερη εμφάνιση των πινάκων
from yellowbrick.cluster import KElbowVisualizer # Παραμένει για το KMeans Elbow

df = pd.read_excel('POS_DATA_BAPR_2024-2025_updated (3).xlsx')
df2 = pd.read_excel('POS_DATA_BAPR_2024-2025_updated (3).xlsx', sheet_name = 'Hierachy Categories & Barcodes')
df3 = pd.read_excel('POS_DATA_BAPR_2024-2025_updated (3).xlsx', sheet_name = 'Loyalty')


#Read the Excel file
#df = pd.read_excel('/content/POS_DATA_BAPR_2024-2025_updated (1).xlsx')
#df2 = pd.read_excel('/content/POS_DATA_BAPR_2024-2025_updated (1).xlsx', sheet_name = 'Hierachy Categories & Barcodes')
#df3 = pd.read_excel('/content/POS_DATA_BAPR_2024-2025_updated (1).xlsx', sheet_name = 'Loyalty')

print(df.head())

In [ ]:
# Βασικός καθαρισμός A, B, C
for col in ["Category A", "Category B", "Category C"]:
    df2[col] = (
        df2[col]
        .astype(str)
        .str.strip()
        .str.title()
    )
    df2[col] = df2[col].replace(["Nan", "None", "Na", ""], np.nan)

df2["CustomCategory"] = df2["Category B"].copy()
# =========================================================
# 3️⃣ ΟΛΕΣ ΟΙ ΠΑΛΙΕΣ ΑΛΛΑΓΕΣ ΣΟΥ ΠΑΝΩ ΣΤΗΝ CustomCategory
# =========================================================

# 3.1 Συσκευασμενο → "Category C + ' σε συσκευασία'"
mask_sysk = df2["CustomCategory"] == "Συσκευασμενο"
df2.loc[mask_sysk, "CustomCategory"] = (
    df2.loc[mask_sysk, "Category C"] + " σε συσκευασία"
)

# 3.2 Merge γαλακτοκομικών σε ενιαία κατηγορία (θα το σπάσουμε μετά)
to_merge_dairy = [
    "Γιαουρτια σε συσκευασία",
    "Τυροκομικα σε συσκευασία",
    "Γαλατα σε συσκευασία",
    "Βουτυρα σε συσκευασία",
    "Κρεμα Γαλακτος σε συσκευασία",
]
df2["CustomCategory"] = df2["CustomCategory"].replace(
    to_merge_dairy, "Γαλακτοκομικά σε συσκευασία"
)

# 3.3 Ρουχων + Ενδυση → Ρούχα & Ενδυση
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Ρουχων", "Ενδυση"], "Ρούχα & Ενδυση"
)

# 3.4 Μπυρες + Κρασια + Οινοπνευματωδη → Οινοπνευματωδη
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Μπυρες", "Κρασια", "Οινοπνευματωδη"], "Οινοπνευματωδη"
)

# 3.5 Σωματος / Ξυριστικα / Χεριων / Προσωπου → Προϊόντα Προσωπικής Φροντίδας
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Σωματος", "Ξυριστικα", "Χεριων", "Προσωπου"],
    "Προϊόντα Προσωπικής Φροντίδας",
)

# 3.6 Βαμβακια / Πανες Ακρατειας → Προιοντα Χαρτου
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Βαμβακια", "Πανες Ακρατειας"], "Προιοντα Χαρτου"
)

# 3.7 Μωρομαντηλα / Πανες Παιδικες → Παιδικα
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Μωρομαντηλα", "Πανες Παιδικες"], "Παιδικα"
)

# 3.8 Βρεφικη Τροφη → Παιδικα
df2["CustomCategory"] = df2["CustomCategory"].replace(
    "Βρεφικη Τροφη", "Παιδικα"
)

# 3.9 Χυμοι / Ροφηματα → Χυμοί & Ροφήματα
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Χυμοι - Τσαι Ψυγειου σε συσκευασία", "Χυμοι", "Ροφηματα σε συσκευασία"],
    "Χυμοί & Ροφήματα",
)
df2["CustomCategory"] = df2["CustomCategory"].replace(
    "Αναψυκτικα", "Χυμοί & Ροφήματα"
)

# 3.10 Κρεας σε συσκευασία → Κατεψυγμενα
df2["CustomCategory"] = df2["CustomCategory"].replace(
    "Κρεας σε συσκευασία", "Κατεψυγμενα"
)

# 3.11 Σαλτσες / Dressings → Σάλτσες & Dressings
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Σαλτσες", "Dressings"], "Σάλτσες & Dressings"
)

# =========================================================
# 4️⃣ ΟΛΕΣ ΟΙ ΝΕΕΣ "ΕΞΥΠΝΕΣ" ΑΛΛΑΓΕΣ ΑΠΟ ΤΗΝ ΑΝΑΛΥΣΗ
# =========================================================

# 4.1 Split Ρούχα & Ενδυση → Προϊόντα Πλυντηρίου Ρούχων vs Ρούχα
laundry_items = [
    "Υγρα Πλυντηριου",
    "Μαλακτικα Πλυντηριου",
    "Ενισχυτικα-Χρωμοπαγιδες",
    "Σκονη Πλυντηριου",
    "Ταμπλετες Πλυντηριου",
    "Αποσκληρυντικα Πλυντηριου",
    "Πλυσιμο Στο Χερι",
    "Σιδερωματος",
]

mask_laundry = (
    (df2["CustomCategory"] == "Ρούχα & Ενδυση")
& (df2["Category C"].isin(laundry_items))
)
df2.loc[mask_laundry, "CustomCategory"] = "Προϊόντα Πλυντηρίου Ρούχων"

# 4.2 Διάλυση "Χυμα" σε λογικές κατηγορίες
xuma_map = {
    "Τυροκομικα": "Γαλακτοκομικά σε συσκευασία",
    "Αλλαντικα": "Αλλαντικα σε συσκευασία",
    "Μαναβικη": "Μαναβικη σε συσκευασία",
    "Ξηροι Καρποι": "Αλμυρα Σνακ",
    "Χαλβας": "Χαλβαδες Ταχινι",
    "Αλιπαστα": "Κονσερβες",
    "Βουτυρα": "Γαλακτοκομικά σε συσκευασία",
}
mask_xuma = df2["Category B"] == "Χυμα"
df2.loc[mask_xuma, "CustomCategory"] = (
    df2.loc[mask_xuma, "Category C"].map(xuma_map).fillna("Χυμα")
)

# 4.3 Αλευρι από Αρτοσκευασματα → Βασικά Υλικά Μαγειρικής
mask_alevri = (df2["Category B"] == "Αρτοσκευασματα") & (df2["Category C"] == "Αλευρι")
df2.loc[mask_alevri, "CustomCategory"] = "Βασικά Υλικά Μαγειρικής"

# 4.4 Σπάσιμο Γαλακτοκομικών σε επιμέρους κατηγορίες
dairy_split_map = {
    "Γιαουρτια": "Γιαουρτια",
    "Τυροκομικα": "Τυροκομικα",
    "Γαλατα": "Γαλατα",
    "Βουτυρα": "Βουτυρα",
    "Κρεμα Γαλακτος": "Κρεμα Γαλακτος",
}
mask_dairy = df2["CustomCategory"] == "Γαλακτοκομικά σε συσκευασία"
df2.loc[mask_dairy, "CustomCategory"] = (
    df2.loc[mask_dairy, "Category C"].map(dairy_split_map).fillna("Γαλακτοκομικά σε συσκευασία")
)

# 4.5 Split Γλυκα Σνακ
mask_glyka = df2["CustomCategory"] == "Γλυκα Σνακ"
c_glyka = df2["Category C"]

df2.loc[mask_glyka & c_glyka.isin(["Μπισκοτα", "Wafer"]), "CustomCategory"] = "Μπισκοτα & Wafers"
df2.loc[mask_glyka & c_glyka.isin(["Σοκολατες", "Ζαχαρωδη"]), "CustomCategory"] = "Σοκολατες & Ζαχαρωδη"
df2.loc[mask_glyka & (c_glyka == "Κρουασαν"), "CustomCategory"] = "Κρουασαν & Bake Snacks"
df2.loc[mask_glyka & c_glyka.isin(["Παραδοσιακα Γλυκισματα", "Κεικ"]), "CustomCategory"] = "Ειδη Ζαχαροπλαστικης"
df2.loc[mask_glyka & (c_glyka == "Τσουρεκι"), "CustomCategory"] = "Αρτοσκευασματα"
df2.loc[mask_glyka & (c_glyka == "Εποχιακα"), "CustomCategory"] = "Εποχιακα Ειδη"

# 4.6 Split Πρωινο
mask_proino = df2["CustomCategory"] == "Πρωινο"
c_pro = df2["Category C"]

# Ροφήματα πρωινού
df2.loc[mask_proino & c_pro.isin(["Καφες", "Τσαι", "Σοκολατουχα Ροφηματα", "Αρωματικα Ροφηματα"]),
         "CustomCategory"] = "Ροφηματα Πρωινου"

# Δημητριακά
df2.loc[mask_proino & (c_pro == "Δημητριακα"), "CustomCategory"] = "Δημητριακα Πρωινου"

# Αλείμματα πρωινού
df2.loc[mask_proino & c_pro.isin(["Μαρμελαδες", "Μελι", "Πραλινες Spreads", "Peanut Butter Spreads"]),
         "CustomCategory"] = "Αλειμματα Πρωινου"

# Εβαπορε → Γαλατα
df2.loc[mask_proino & (c_pro == "Εβαπορε"), "CustomCategory"] = "Γαλατα"

# 4.7 Split Χυμοί & Ροφήματα
mask_drinks = df2["CustomCategory"] == "Χυμοί & Ροφήματα"
c_dr = df2["Category C"]

# Αναψυκτικά
df2.loc[mask_drinks & c_dr.isin(
    ["Cola", "Πορτοκαλαδα", "Λεμοναδα", "Γκαζοζα", "Soda Tonik Mixers", "Various Flavours"]
), "CustomCategory"] = "Αναψυκτικα"

# Χυμοί & Νέκταρ
df2.loc[mask_drinks & c_dr.isin(
    ["Φυσικοι", "Νεκταρ", "Φρουτοποτα", "Συμπυκνωμενοι"]
), "CustomCategory"] = "Χυμοι & Νεκταρ"

# Έτοιμα ροφήματα (Ice Tea, Ice Coffee κ.λπ.)
df2.loc[mask_drinks & c_dr.isin(
    ["Ice Tea Ice Coffee", "Ροφηματα", "Χυμοι - Τσαι Ψυγειου"]
), "CustomCategory"] = "Rtd Ροφηματα"

# Ενεργειακά
df2.loc[mask_drinks & (c_dr == "Ενεργειακα Ισοτονικα"), "CustomCategory"] = "Ενεργειακα & Ισοτονικα"

# 4.8 Split Κατεψυγμενα
mask_frozen = df2["CustomCategory"] == "Κατεψυγμενα"
c_fr = df2["Category C"]

df2.loc[mask_frozen & (c_fr == "Παγωτα"), "CustomCategory"] = "Κατεψυγμενα Παγωτα"
df2.loc[mask_frozen & (c_fr == "Ζυμες"), "CustomCategory"] = "Κατεψυγμενες Ζυμες"
df2.loc[mask_frozen & (c_fr == "Ψαρικα"), "CustomCategory"] = "Κατεψυγμενα Ψαρικα"
df2.loc[mask_frozen & (c_fr == "Λαχανικα"), "CustomCategory"] = "Κατεψυγμενα Λαχανικα"
df2.loc[mask_frozen & c_fr.isin(["Κρεας", "Ετοιμα Φαγητα"]), "CustomCategory"] = "Κατεψυγμενα Κρεας & Γευματα"

# 4.9 Split Αλμυρα Σνακ → Ξηροι Καρποι ξεχωριστά
mask_salty = df2["CustomCategory"] == "Αλμυρα Σνακ"
c_salty = df2["Category C"]

df2.loc[mask_salty & (c_salty == "Ξηροι Καρποι"), "CustomCategory"] = "Ξηροι Καρποι"

# =========================================================
# 5️⃣ Κανόνας: μικρές κατηγορίες (<10 barcodes) → "Διαφορα"
# =========================================================
counts = df2["CustomCategory"].value_counts()
small_cats = counts[counts < 10].index.tolist()
df2["CustomCategory"] = df2["CustomCategory"].replace(small_cats, "Διαφορα")

# Ζυμες Ψυγειου σε συσκευασία → Αρτοσκευασματα
df2.loc[df2["CustomCategory"] == "Ζυμες Ψυγειου σε συσκευασία", "CustomCategory"] = "Αρτοσκευασματα"

# =========================================================
# 6️⃣ Γρήγορος έλεγχος
# =========================================================
print("Μοναδικές Category B       :", df2["Category B"].nunique())
print("Μοναδικές CustomCategory   :", df2["CustomCategory"].nunique())
print("\nTop 40 CustomCategory:")
print(df2["CustomCategory"].value_counts().head(40))

# Νέα ενότητα

In [ ]:
#exclude quantities < 1
print(df.shape)
df = df[df['Quantity'] >= 1]
print(df.shape)

In [ ]:
display(df.describe())

In [ ]:
#exclude non positive values
df = df[df['Value_'] > 0]
print(df.shape)

In [ ]:
#exclude non integer quantities
df = df[df['Quantity'] % 1 == 0]
print(df.shape)

In [ ]:
#create new column Price
df['Price'] = df['Value_'] / df['Quantity']
print(df.shape)

In [ ]:
#Remove baskets with no LoyaltyCard attached
df = df[df['LoyaltyCard_ID'].notna()]
print(df.shape)


In [ ]:
df = df[df['Value_'].notna()]
df = df[df['Barcode'].notna()]
df = df[df['Date_'].notna()]
df = df[df['Basket_ID'].notna()]
df = df[df['Quantity'].notna()]
print(df.shape)
df.head()

In [ ]:
#Exclude barcodes that are not contained in the list of real barcodes
df = df[df['Barcode'].isin(df2['Barcode'])]
print(df.shape)

In [ ]:
#Exclude transactions that did not contain cardid's where cardholder was known or his Status was na
# try to fix the na into NaN?
#valid_cards = df3.loc[df3['Status'].str.contains('na', na=False), 'Cardholder']

#df = df[~df['LoyaltyCard_ID'].isin(valid_cards)]Fsi
#print(df.shape)

In [ ]:
#Convert date from string to datetime
df['Date_'] = pd.to_datetime(df['Date_'], errors='coerce', dayfirst=True)
df.head()

In [ ]:
df.replace(['na'], pd.NA, inplace=True)
df.describe()


In [ ]:
Q1 = df['Value_'].quantile(0.25)
Q3 = df['Value_'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter the DataFrame
df = df[(df['Value_'] >= lower_bound) & (df['Value_'] <= upper_bound)]
print(df.shape)

In [ ]:
plt.hist(df['Value_'], bins=20, edgecolor='black')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title('Distribution of Value')
plt.show()

In [ ]:
df.head()

In [ ]:
df2.head()

In [ ]:
df3.head()

In [ ]:
df = pd.merge(df, df3, left_on='LoyaltyCard_ID', right_on='Cardholder', how='inner')
df.head()

In [ ]:
# [ΠΕΡΙΠΟΥ ΓΡΑΜΜΗ 160]
df = pd.merge(df, df3, left_on='LoyaltyCard_ID', right_on='Cardholder', how='inner')
df.head()
# --- ΚΑΘΑΡΟ MERGE DF ΜΕ CUSTOM CATEGORY ---
# Προετοιμάζουμε το df_clean κρατώντας μόνο τις βασικές στήλες συναλλαγής
df.rename(columns={"Value_": "Value"}, inplace=True) 
df_clean = df[['Basket_ID', 'Value', 'Quantity', 'Barcode']].copy()

# Εισάγουμε μόνο τις στήλες Barcode και CustomCategory από το df2
df_clean = pd.merge(df_clean, df2[['Barcode', 'CustomCategory']], on='Barcode', how='inner')

# Αφαίρεση NaT/NaN
df_clean.dropna(subset=['CustomCategory', 'Basket_ID', 'Value'], inplace=True)
print(f"Διαστάσεις DataFrame μετά το Clean/Merge: {df_clean.shape}")


In [ ]:
file_path = r"C:\Users\pc\Downloads\POS_DATA_BAPR_2024-2025_updated (1).xlsx"
hier = pd.read_excel('POS_DATA_BAPR_2024-2025_updated (3).xlsx', sheet_name = 'Hierachy Categories & Barcodes')
 
# Βασικός καθαρισμός A, B, C
for col in ["Category A", "Category B", "Category C"]:
    hier[col] = (
        hier[col]
        .astype(str)
        .str.strip()
        .str.title()
    )
    hier[col] = hier[col].replace(["Nan", "None", "Na", ""], np.nan)
 
# 2️⃣ Δημιουργούμε νέα στήλη CustomCategory (ξεκινάει ίδια με Category B)
hier["CustomCategory"] = hier["Category B"].copy()
 
# =========================================================
# 3️⃣ ΟΛΕΣ ΟΙ ΠΑΛΙΕΣ ΑΛΛΑΓΕΣ ΣΟΥ ΠΑΝΩ ΣΤΗΝ CustomCategory
# =========================================================
 
# 3.1 Συσκευασμενο → "Category C + ' σε συσκευασία'"
mask_sysk = hier["CustomCategory"] == "Συσκευασμενο"
hier.loc[mask_sysk, "CustomCategory"] = (
    hier.loc[mask_sysk, "Category C"] + " σε συσκευασία"
)
 
# 3.2 Merge γαλακτοκομικών σε ενιαία κατηγορία (θα το σπάσουμε μετά)
to_merge_dairy = [
    "Γιαουρτια σε συσκευασία",
    "Τυροκομικα σε συσκευασία",
    "Γαλατα σε συσκευασία",
    "Βουτυρα σε συσκευασία",
    "Κρεμα Γαλακτος σε συσκευασία",
]
hier["CustomCategory"] = hier["CustomCategory"].replace(
    to_merge_dairy, "Γαλακτοκομικά σε συσκευασία"
)
 
# 3.3 Ρουχων + Ενδυση → Ρούχα & Ενδυση
hier["CustomCategory"] = hier["CustomCategory"].replace(
    ["Ρουχων", "Ενδυση"], "Ρούχα & Ενδυση"
)
 
# 3.4 Μπυρες + Κρασια + Οινοπνευματωδη → Οινοπνευματωδη
hier["CustomCategory"] = hier["CustomCategory"].replace(
    ["Μπυρες", "Κρασια", "Οινοπνευματωδη"], "Οινοπνευματωδη"
)
 
# 3.5 Σωματος / Ξυριστικα / Χεριων / Προσωπου → Προϊόντα Προσωπικής Φροντίδας
hier["CustomCategory"] = hier["CustomCategory"].replace(
    ["Σωματος", "Ξυριστικα", "Χεριων", "Προσωπου"],
    "Προϊόντα Προσωπικής Φροντίδας",
)
 
# 3.6 Βαμβακια / Πανες Ακρατειας → Προιοντα Χαρτου
hier["CustomCategory"] = hier["CustomCategory"].replace(
    ["Βαμβακια", "Πανες Ακρατειας"], "Προιοντα Χαρτου"
)
 
# 3.7 Μωρομαντηλα / Πανες Παιδικες → Παιδικα
hier["CustomCategory"] = hier["CustomCategory"].replace(
    ["Μωρομαντηλα", "Πανες Παιδικες"], "Παιδικα"
)
 
# 3.8 Βρεφικη Τροφη → Παιδικα
hier["CustomCategory"] = hier["CustomCategory"].replace(
    "Βρεφικη Τροφη", "Παιδικα"
)
 
# 3.9 Χυμοι / Ροφηματα → Χυμοί & Ροφήματα
hier["CustomCategory"] = hier["CustomCategory"].replace(
    ["Χυμοι - Τσαι Ψυγειου σε συσκευασία", "Χυμοι", "Ροφηματα σε συσκευασία"],
    "Χυμοί & Ροφήματα",
)
hier["CustomCategory"] = hier["CustomCategory"].replace(
    "Αναψυκτικα", "Χυμοί & Ροφήματα"
)
 
# 3.10 Κρεας σε συσκευασία → Κατεψυγμενα
hier["CustomCategory"] = hier["CustomCategory"].replace(
    "Κρεας σε συσκευασία", "Κατεψυγμενα"
)
 
# 3.11 Σαλτσες / Dressings → Σάλτσες & Dressings
hier["CustomCategory"] = hier["CustomCategory"].replace(
    ["Σαλτσες", "Dressings"], "Σάλτσες & Dressings"
)
 
# =========================================================
# 4️⃣ ΟΛΕΣ ΟΙ ΝΕΕΣ "ΕΞΥΠΝΕΣ" ΑΛΛΑΓΕΣ ΑΠΟ ΤΗΝ ΑΝΑΛΥΣΗ
# =========================================================
 
# 4.1 Split Ρούχα & Ενδυση → Προϊόντα Πλυντηρίου Ρούχων vs Ρούχα
laundry_items = [
    "Υγρα Πλυντηριου",
    "Μαλακτικα Πλυντηριου",
    "Ενισχυτικα-Χρωμοπαγιδες",
    "Σκονη Πλυντηριου",
    "Ταμπλετες Πλυντηριου",
    "Αποσκληρυντικα Πλυντηριου",
    "Πλυσιμο Στο Χερι",
    "Σιδερωματος",
]
 
mask_laundry = (
    (hier["CustomCategory"] == "Ρούχα & Ενδυση")
& (hier["Category C"].isin(laundry_items))
)
hier.loc[mask_laundry, "CustomCategory"] = "Προϊόντα Πλυντηρίου Ρούχων"
 
# 4.2 Διάλυση "Χυμα" σε λογικές κατηγορίες
xuma_map = {
    "Τυροκομικα": "Γαλακτοκομικά σε συσκευασία",
    "Αλλαντικα": "Αλλαντικα σε συσκευασία",
    "Μαναβικη": "Μαναβικη σε συσκευασία",
    "Ξηροι Καρποι": "Αλμυρα Σνακ",
    "Χαλβας": "Χαλβαδες Ταχινι",
    "Αλιπαστα": "Κονσερβες",
    "Βουτυρα": "Γαλακτοκομικά σε συσκευασία",
}
mask_xuma = hier["Category B"] == "Χυμα"
hier.loc[mask_xuma, "CustomCategory"] = (
    hier.loc[mask_xuma, "Category C"].map(xuma_map).fillna("Χυμα")
)
 
# 4.3 Αλευρι από Αρτοσκευασματα → Βασικά Υλικά Μαγειρικής
mask_alevri = (hier["Category B"] == "Αρτοσκευασματα") & (hier["Category C"] == "Αλευρι")
hier.loc[mask_alevri, "CustomCategory"] = "Βασικά Υλικά Μαγειρικής"
 
# 4.4 Σπάσιμο Γαλακτοκομικών σε επιμέρους κατηγορίες
dairy_split_map = {
    "Γιαουρτια": "Γιαουρτια",
    "Τυροκομικα": "Τυροκομικα",
    "Γαλατα": "Γαλατα",
    "Βουτυρα": "Βουτυρα",
    "Κρεμα Γαλακτος": "Κρεμα Γαλακτος",
}
mask_dairy = hier["CustomCategory"] == "Γαλακτοκομικά σε συσκευασία"
hier.loc[mask_dairy, "CustomCategory"] = (
    hier.loc[mask_dairy, "Category C"].map(dairy_split_map).fillna("Γαλακτοκομικά σε συσκευασία")
)
 
# 4.5 Split Γλυκα Σνακ
mask_glyka = hier["CustomCategory"] == "Γλυκα Σνακ"
c_glyka = hier["Category C"]
 
hier.loc[mask_glyka & c_glyka.isin(["Μπισκοτα", "Wafer"]), "CustomCategory"] = "Μπισκοτα & Wafers"
hier.loc[mask_glyka & c_glyka.isin(["Σοκολατες", "Ζαχαρωδη"]), "CustomCategory"] = "Σοκολατες & Ζαχαρωδη"
hier.loc[mask_glyka & (c_glyka == "Κρουασαν"), "CustomCategory"] = "Κρουασαν & Bake Snacks"
hier.loc[mask_glyka & c_glyka.isin(["Παραδοσιακα Γλυκισματα", "Κεικ"]), "CustomCategory"] = "Ειδη Ζαχαροπλαστικης"
hier.loc[mask_glyka & (c_glyka == "Τσουρεκι"), "CustomCategory"] = "Αρτοσκευασματα"
hier.loc[mask_glyka & (c_glyka == "Εποχιακα"), "CustomCategory"] = "Εποχιακα Ειδη"
 
# 4.6 Split Πρωινο
mask_proino = hier["CustomCategory"] == "Πρωινο"
c_pro = hier["Category C"]
 
# Ροφήματα πρωινού
hier.loc[mask_proino & c_pro.isin(["Καφες", "Τσαι", "Σοκολατουχα Ροφηματα", "Αρωματικα Ροφηματα"]),
         "CustomCategory"] = "Ροφηματα Πρωινου"
 
# Δημητριακά
hier.loc[mask_proino & (c_pro == "Δημητριακα"), "CustomCategory"] = "Δημητριακα Πρωινου"
 
# Αλείμματα πρωινού
hier.loc[mask_proino & c_pro.isin(["Μαρμελαδες", "Μελι", "Πραλινες Spreads", "Peanut Butter Spreads"]),
         "CustomCategory"] = "Αλειμματα Πρωινου"
 
# Εβαπορε → Γαλατα
hier.loc[mask_proino & (c_pro == "Εβαπορε"), "CustomCategory"] = "Γαλατα"
 
# 4.7 Split Χυμοί & Ροφήματα
mask_drinks = hier["CustomCategory"] == "Χυμοί & Ροφήματα"
c_dr = hier["Category C"]
 
# Αναψυκτικά
hier.loc[mask_drinks & c_dr.isin(
    ["Cola", "Πορτοκαλαδα", "Λεμοναδα", "Γκαζοζα", "Soda Tonik Mixers", "Various Flavours"]
), "CustomCategory"] = "Αναψυκτικα"
 
# Χυμοί & Νέκταρ
hier.loc[mask_drinks & c_dr.isin(
    ["Φυσικοι", "Νεκταρ", "Φρουτοποτα", "Συμπυκνωμενοι"]
), "CustomCategory"] = "Χυμοι & Νεκταρ"
 
# Έτοιμα ροφήματα (Ice Tea, Ice Coffee κ.λπ.)
hier.loc[mask_drinks & c_dr.isin(
    ["Ice Tea Ice Coffee", "Ροφηματα", "Χυμοι - Τσαι Ψυγειου"]
), "CustomCategory"] = "Rtd Ροφηματα"
 
# Ενεργειακά
hier.loc[mask_drinks & (c_dr == "Ενεργειακα Ισοτονικα"), "CustomCategory"] = "Ενεργειακα & Ισοτονικα"
 
# 4.8 Split Κατεψυγμενα
mask_frozen = hier["CustomCategory"] == "Κατεψυγμενα"
c_fr = hier["Category C"]
 
hier.loc[mask_frozen & (c_fr == "Παγωτα"), "CustomCategory"] = "Κατεψυγμενα Παγωτα"
hier.loc[mask_frozen & (c_fr == "Ζυμες"), "CustomCategory"] = "Κατεψυγμενες Ζυμες"
hier.loc[mask_frozen & (c_fr == "Ψαρικα"), "CustomCategory"] = "Κατεψυγμενα Ψαρικα"
hier.loc[mask_frozen & (c_fr == "Λαχανικα"), "CustomCategory"] = "Κατεψυγμενα Λαχανικα"
hier.loc[mask_frozen & c_fr.isin(["Κρεας", "Ετοιμα Φαγητα"]), "CustomCategory"] = "Κατεψυγμενα Κρεας & Γευματα"
 
# 4.9 Split Αλμυρα Σνακ → Ξηροι Καρποι ξεχωριστά
mask_salty = hier["CustomCategory"] == "Αλμυρα Σνακ"
c_salty = hier["Category C"]
 
hier.loc[mask_salty & (c_salty == "Ξηροι Καρποι"), "CustomCategory"] = "Ξηροι Καρποι"
 
# =========================================================
# 5️⃣ Κανόνας: μικρές κατηγορίες (<10 barcodes) → "Διαφορα"
# =========================================================
counts = hier["CustomCategory"].value_counts()
small_cats = counts[counts < 10].index.tolist()
hier["CustomCategory"] = hier["CustomCategory"].replace(small_cats, "Διαφορα")
 
# =========================================================
# 6️⃣ Γρήγορος έλεγχος
# =========================================================
print("Μοναδικές Category B       :", hier["Category B"].nunique())
print("Μοναδικές CustomCategory   :", hier["CustomCategory"].nunique())
print("\nTop 40 CustomCategory:")
print(hier["CustomCategory"].value_counts().head(40))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# =============================================================================
# 1. ΦΟΡΤΩΣΗ ΔΕΔΟΜΕΝΩΝ
# =============================================================================
print("--- 1. Φόρτωση Δεδομένων ---")
# Φόρτωση από τα CSV αρχεία (προσαρμόστε τα ονόματα αν τρέχετε τοπικά)
df = pd.read_excel('POS_DATA_BAPR_2024-2025_updated (3).xlsx', sheet_name='POS Data')
df2 = pd.read_excel('POS_DATA_BAPR_2024-2025_updated (3).xlsx', sheet_name='Hierachy Categories & Barcodes')

# =============================================================================
# 2. ΔΗΜΙΟΥΡΓΙΑ CUSTOM CATEGORY (ΟΛΟΙ ΟΙ ΚΑΝΟΝΕΣ)
# =============================================================================
print("--- 2. Εφαρμογή Κανόνων Custom Category ---")

# Βασικός καθαρισμός strings
for col in ["Category A", "Category B", "Category C"]:
    df2[col] = df2[col].astype(str).str.strip().str.title().replace(["Nan", "None", "Na", "", "nan"], np.nan)

df2["CustomCategory"] = df2["Category B"].copy()

# --- Εφαρμογή των κανόνων 3.1 έως 4.9 ---

# 3.1 Συσκευασμενο
mask_sysk = df2["CustomCategory"] == "Συσκευασμενο"
df2.loc[mask_sysk, "CustomCategory"] = df2.loc[mask_sysk, "Category C"] + " σε συσκευασία"

# 3.2 Merge γαλακτοκομικών
to_merge_dairy = ["Γιαουρτια σε συσκευασία", "Τυροκομικα σε συσκευασία", "Γαλατα σε συσκευασία", "Βουτυρα σε συσκευασία", "Κρεμα Γαλακτος σε συσκευασία"]
df2["CustomCategory"] = df2["CustomCategory"].replace(to_merge_dairy, "Γαλακτοκομικά σε συσκευασία")

# 3.3 - 3.11 Διάφορες αντικαταστάσεις
df2["CustomCategory"] = df2["CustomCategory"].replace(["Ρουχων", "Ενδυση"], "Ρούχα & Ενδυση")
df2["CustomCategory"] = df2["CustomCategory"].replace(["Μπυρες", "Κρασια", "Οινοπνευματωδη"], "Οινοπνευματωδη")
df2["CustomCategory"] = df2["CustomCategory"].replace(["Σωματος", "Ξυριστικα", "Χεριων", "Προσωπου"], "Προϊόντα Προσωπικής Φροντίδας")
df2["CustomCategory"] = df2["CustomCategory"].replace(["Βαμβακια", "Πανες Ακρατειας"], "Προιοντα Χαρτου")
df2["CustomCategory"] = df2["CustomCategory"].replace(["Μωρομαντηλα", "Πανες Παιδικες", "Βρεφικη Τροφη"], "Παιδικα")
df2["CustomCategory"] = df2["CustomCategory"].replace(["Χυμοι - Τσαι Ψυγειου σε συσκευασία", "Χυμοι", "Ροφηματα σε συσκευασία", "Αναψυκτικα"], "Χυμοί & Ροφήματα")
df2["CustomCategory"] = df2["CustomCategory"].replace("Κρεας σε συσκευασία", "Κατεψυγμενα")
df2["CustomCategory"] = df2["CustomCategory"].replace(["Σαλτσες", "Dressings"], "Σάλτσες & Dressings")

# 4.1 Laundry
laundry_items = ["Υγρα Πλυντηριου", "Μαλακτικα Πλυντηριου", "Ενισχυτικα-Χρωμοπαγιδες", "Σκονη Πλυντηριου", "Ταμπλετες Πλυντηριου", "Αποσκληρυντικα Πλυντηριου", "Πλυσιμο Στο Χερι", "Σιδερωματος"]
mask_laundry = (df2["CustomCategory"] == "Ρούχα & Ενδυση") & (df2["Category C"].isin(laundry_items))
df2.loc[mask_laundry, "CustomCategory"] = "Προϊόντα Πλυντηρίου Ρούχων"

# 4.2 Χύμα breakdown
xuma_map = {
    "Τυροκομικα": "Γαλακτοκομικά σε συσκευασία", "Αλλαντικα": "Αλλαντικα σε συσκευασία",
    "Μαναβικη": "Μαναβικη σε συσκευασία", "Ξηροι Καρποι": "Αλμυρα Σνακ",
    "Χαλβας": "Χαλβαδες Ταχινι", "Αλιπαστα": "Κονσερβες", "Βουτυρα": "Γαλακτοκομικά σε συσκευασία"
}
mask_xuma = df2["Category B"] == "Χυμα"
df2.loc[mask_xuma, "CustomCategory"] = df2.loc[mask_xuma, "Category C"].map(xuma_map).fillna("Χυμα")

# 4.3 Αλεύρι
mask_alevri = (df2["Category B"] == "Αρτοσκευασματα") & (df2["Category C"] == "Αλευρι")
df2.loc[mask_alevri, "CustomCategory"] = "Βασικά Υλικά Μαγειρικής"

# 4.4 Split Dairy (Σημαντικό!)
dairy_split_map = {"Γιαουρτια": "Γιαουρτια", "Τυροκομικα": "Τυροκομικα", "Γαλατα": "Γαλατα", "Βουτυρα": "Βουτυρα", "Κρεμα Γαλακτος": "Κρεμα Γαλακτος"}
mask_dairy = df2["CustomCategory"] == "Γαλακτοκομικά σε συσκευασία"
df2.loc[mask_dairy, "CustomCategory"] = df2.loc[mask_dairy, "Category C"].map(dairy_split_map).fillna("Γαλακτοκομικά σε συσκευασία")

# 4.5 - 4.9 (Snacks, Breakfast, Drinks, Frozen, Salty)
mask_glyka = df2["CustomCategory"] == "Γλυκα Σνακ"
df2.loc[mask_glyka & df2["Category C"].isin(["Μπισκοτα", "Wafer"]), "CustomCategory"] = "Μπισκοτα & Wafers"
df2.loc[mask_glyka & df2["Category C"].isin(["Σοκολατες", "Ζαχαρωδη"]), "CustomCategory"] = "Σοκολατες & Ζαχαρωδη"
df2.loc[mask_glyka & (df2["Category C"] == "Κρουασαν"), "CustomCategory"] = "Κρουασαν & Bake Snacks"

mask_proino = df2["CustomCategory"] == "Πρωινο"
df2.loc[mask_proino & df2["Category C"].isin(["Καφες", "Τσαι"]), "CustomCategory"] = "Ροφηματα Πρωινου"
df2.loc[mask_proino & (df2["Category C"] == "Δημητριακα"), "CustomCategory"] = "Δημητριακα Πρωινου"
df2.loc[mask_proino & (df2["Category C"] == "Εβαπορε"), "CustomCategory"] = "Γαλατα"

mask_frozen = df2["CustomCategory"] == "Κατεψυγμενα"
df2.loc[mask_frozen & (df2["Category C"] == "Παγωτα"), "CustomCategory"] = "Κατεψυγμενα Παγωτα"

mask_salty = df2["CustomCategory"] == "Αλμυρα Σνακ"
df2.loc[mask_salty & (df2["Category C"] == "Ξηροι Καρποι"), "CustomCategory"] = "Ξηροι Καρποι"

# 5. Μικρές κατηγορίες -> Διάφορα
counts = df2["CustomCategory"].value_counts()
df2["CustomCategory"] = df2["CustomCategory"].replace(counts[counts < 10].index.tolist(), "Διαφορα")

print("Custom Categories δημιουργήθηκαν.")

# =============================================================================
# 3. ΠΡΟΕΤΟΙΜΑΣΙΑ ΣΥΝΑΛΛΑΓΩΝ & MERGE
# =============================================================================
df.rename(columns={"Value_": "Value"}, inplace=True)
df = df[df['Quantity'] >= 1]
df = df[df['Value'] > 0]
df = df.dropna(subset=['Value', 'Barcode', 'Basket_ID'])

# Merge: Εδώ συνδέουμε το POS Data με το CustomCategory που φτιάξαμε
df_clean = pd.merge(df[['Basket_ID', 'Value', 'Barcode']], df2[['Barcode', 'CustomCategory']], on='Barcode', how='inner')

# =============================================================================
# 4. CLUSTERING (K-MEANS)
# =============================================================================
print("--- 3. Εκτέλεση Basket Clustering ---")

# Pivot: Καλάθια x CustomCategories
basket_pivot = df_clean.pivot_table(index='Basket_ID', columns='CustomCategory', values='Value', aggfunc='sum').fillna(0)
# Μετατροπή σε ποσοστά
basket_pct = basket_pivot.div(basket_pivot.sum(axis=1), axis=0)

# K-Means (k=5)
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
basket_pct['Cluster'] = kmeans.fit_predict(basket_pct)

# =============================================================================
# 5. VISUALIZATIONS
# =============================================================================
print("--- 4. Οπτικοποίηση Αποτελεσμάτων ---")

# A. Μέγεθος Clusters
plt.figure(figsize=(8, 4))
basket_pct['Cluster'].value_counts().sort_index().plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Μέγεθος Basket Clusters')
plt.xlabel('Cluster')
plt.ylabel('Αριθμός Καλαθιών')
plt.show()

# B. Heatmap (Το προφίλ των Clusters)
cluster_centers = basket_pct.groupby('Cluster').mean()
top_cats = basket_pct.drop('Cluster', axis=1).mean().sort_values(ascending=False).head(15).index

plt.figure(figsize=(12, 8))
sns.heatmap(cluster_centers[top_cats].T, annot=True, cmap='YlGnBu', fmt='.2f')
plt.title('Προφίλ Clusters (Βάσει CustomCategory)')
plt.xlabel('Cluster')
plt.ylabel('CustomCategory')
plt.show()

# C. PCA Plot
pca = PCA(n_components=2)
coords = pca.fit_transform(basket_pct.drop('Cluster', axis=1))
plt.figure(figsize=(8, 6))
plt.scatter(coords[:, 0], coords[:, 1], c=basket_pct['Cluster'], cmap='viridis', alpha=0.5)
plt.title('PCA Basket Clusters')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.colorbar(label='Cluster')
plt.show()

# =============================================================================
# 6. ΕΡΜΗΝΕΙΑ CLUSTERS
# =============================================================================
print("\n--- 5. Ερμηνεία Clusters (Top Categories) ---")
for i in range(5):
    print(f"\nCluster {i}:")
    top_3 = cluster_centers.loc[i].sort_values(ascending=False).head(3)
    for cat, val in top_3.items():
        print(f"  - {cat}: {val*100:.1f}%")

Ανάλυση Ομάδων (Basket Segments)
Με βάση τα κέντρα των ομάδων (Cluster Centers), εντοπίσαμε 5 ξεκάθαρους τύπους καλαθιών:

Cluster 0: "Snacking & Treats" (Καλάθι Σνακ)
Χαρακτηριστικά: Το κυρίαρχο προϊόν είναι τα Μπισκότα & Wafers (>54% της αξίας του καλαθιού), ακολουθούμενα από Σοκολάτες & Ζαχαρώδη.

Ερμηνεία: Πρόκειται για στοχευμένες αγορές "απόλαυσης" ή σνακ, πιθανότατα μικρά καλάθια που καλύπτουν μια άμεση ανάγκη για γλυκό ή κολατσιό, και όχι το κύριο φαγητό της ημέρας.

Cluster 1: "The Refill / Variety Basket" (Καλάθι Ποικιλίας / Αναπλήρωσης)
Χαρακτηριστικά: Δεν υπάρχει κυρίαρχη κατηγορία. Το καλάθι είναι "πολυσυλλεκτικό" με μικρά ποσοστά από Προϊόντα Χάρτου, Γιαούρτια, Ροφήματα, Αναψυκτικά κ.λπ.

Ερμηνεία: Αυτό είναι το πιο ενδιαφέρον cluster. Αντιπροσωπεύει τα ολοκληρωμένα ψώνια (stock-up) ή την αναπλήρωση του νοικοκυριού, όπου ο πελάτης αγοράζει λίγο από όλα. Είναι οι πιο "κανονικοί" αγοραστές σούπερ μάρκετ.

Cluster 2: "The Bakery Run" (Καλάθι Φούρνου)
Χαρακτηριστικά: Κυριαρχούν τα Αρτοσκευάσματα (>52% της αξίας).

Ερμηνεία: Πελάτες που επισκέπτονται το κατάστημα κυρίως ως φούρνο. Αγοράζουν ψωμί, φρυγανιές ή πίτες. Συχνά συνδυάζεται με λίγο γάλα, αλλά ο κύριος σκοπός επίσκεψης είναι το ψωμί.

Cluster 3: "Breakfast & Deli" (Καλάθι Τυροκομικών/Πρωινού)
Χαρακτηριστικά: Πολύ ισχυρή παρουσία σε Τυροκομικά (>45%) και Αλλαντικά.

Ερμηνεία: Αγορές που εστιάζουν στο "αλμυρό" πρωινό ή το βραδινό σνακ (τοστ/σάντουιτς). Είναι μια ομάδα που έρχεται για τα φρέσκα προϊόντα ψυγείου.

Cluster 4: "Daily Essentials / Milk Run" (Καλάθι Γάλακτος)
Χαρακτηριστικά: Το Γάλα αποτελεί πάνω από το 56% της αξίας του καλαθιού.

Ερμηνεία: Η πιο κλασική "γρήγορη" επίσκεψη στο σούπερ μάρκετ. Ο πελάτης μπήκε μόνο (ή κυρίως) για να πάρει γάλα που του τελείωσε. Είναι πολύ συχνή αποστολή (shopping mission).

In [ ]:
# Δημιουργία περιγραφών για κάθε cluster
cluster_names = {
    0: "Cluster 0: Snacking & Treats (Μπισκότα/Σοκολάτες)",
    1: "Cluster 1: Variety / Refill (Ποικιλία/Χαρτικά/Γενικό)",
    2: "Cluster 2: Bakery Run (Αρτοσκευάσματα)",
    3: "Cluster 3: Cheese & Deli (Τυροκομικά/Αλλαντικά)",
    4: "Cluster 4: Daily Essentials (Γάλα)"
}

# Εκτύπωση ανάλυσης
cluster_centers = basket_pct.groupby('Cluster').mean()

for i in range(5):
    print(f"\n{cluster_names[i]}")
    print("Top 3 Κατηγορίες (Μέσο % στο καλάθι):")
    top_3 = cluster_centers.loc[i].sort_values(ascending=False).head(3)
    for cat, val in top_3.items():
        print(f"  - {cat}: {val*100:.1f}%")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# =============================================================================
# 1. ΦΟΡΤΩΣΗ & ΚΑΘΑΡΙΣΜΟΣ
# =============================================================================
print("--- 1. Φόρτωση & Καθαρισμός ---")

df = pd.read_excel('POS_DATA_BAPR_2024-2025_updated (3).xlsx', sheet_name='POS Data')
df2 = pd.read_excel('POS_DATA_BAPR_2024-2025_updated (3).xlsx', sheet_name='Hierachy Categories & Barcodes')
df3 = pd.read_excel('POS_DATA_BAPR_2024-2025_updated (3).xlsx', sheet_name='Loyalty')

# Καθαρισμός Κατηγοριών
for col in ["Category A", "Category B", "Category C"]:
    df2[col] = df2[col].astype(str).str.strip().str.title().replace(["Nan", "None", "Na", "", "nan"], np.nan)
df2["CustomCategory"] = df2["Category B"].copy()

# Κανόνες Καθαρισμού
mask_sysk = df2["CustomCategory"] == "Συσκευασμενο"
df2.loc[mask_sysk, "CustomCategory"] = df2.loc[mask_sysk, "Category C"] + " σε συσκευασία"
to_merge_dairy = ["Γιαουρτια σε συσκευασία", "Τυροκομικα σε συσκευασία", "Γαλατα σε συσκευασία", "Βουτυρα σε συσκευασία", "Κρεμα Γαλακτος σε συσκευασία"]
df2["CustomCategory"] = df2["CustomCategory"].replace(to_merge_dairy, "Γαλακτοκομικά σε συσκευασία")
df2["CustomCategory"] = df2["CustomCategory"].replace(["Ρουχων", "Ενδυση"], "Ρούχα & Ενδυση")
df2["CustomCategory"] = df2["CustomCategory"].replace(["Μπυρες", "Κρασια", "Οινοπνευματωδη"], "Οινοπνευματωδη")
counts = df2["CustomCategory"].value_counts()
df2["CustomCategory"] = df2["CustomCategory"].replace(counts[counts < 10].index.tolist(), "Διαφορα")

# Καθαρισμός Συναλλαγών
df.rename(columns={"Value_": "Value"}, inplace=True)
df = df[df['Quantity'] >= 1]
df = df[df['Value'] > 0]
df['Date_'] = pd.to_datetime(df['Date_'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['Value', 'Barcode', 'Basket_ID', 'Date_', 'LoyaltyCard_ID'])

# Merge
df_clean = pd.merge(df, df2[['Barcode', 'CustomCategory']], on='Barcode', how='inner')

# =============================================================================
# 2. BASKET SEGMENTATION (k=5)
# =============================================================================
print("\n--- 2. Basket Segmentation ---")
basket_pivot = df_clean.pivot_table(index='Basket_ID', columns='CustomCategory', values='Value', aggfunc='sum').fillna(0)
basket_pct = basket_pivot.div(basket_pivot.sum(axis=1), axis=0)

kmeans_basket = KMeans(n_clusters=5, random_state=42, n_init=10)
basket_pct['Basket_Cluster'] = kmeans_basket.fit_predict(basket_pct)

# Ενσωμάτωση basket cluster στο df_clean
basket_clusters_map = basket_pct[['Basket_Cluster']].reset_index()
df_clean = pd.merge(df_clean, basket_clusters_map, on='Basket_ID', how='inner')

# =============================================================================
# 3. ΠΡΟΕΤΟΙΜΑΣΙΑ ΓΙΑ CUSTOMER SEGMENTATION
# =============================================================================
print("\n--- 3. Προετοιμασία Customer Data ---")

# Basket Level Aggregation
basket_summary = df_clean.groupby('Basket_ID').agg({
    'LoyaltyCard_ID': 'first',
    'Date_': 'max',
    'Value': 'sum',
    'Basket_Cluster': 'first'
}).reset_index()

# One-Hot Encoding Basket Types
basket_dummies = pd.get_dummies(basket_summary['Basket_Cluster'], prefix='Basket_Type')
basket_summary = pd.concat([basket_summary, basket_dummies], axis=1)

# Customer Level Aggregation
ref_date = basket_summary['Date_'].max()
basket_type_cols = basket_dummies.columns.tolist()
agg_rules = {
    'Value': ['sum', 'count', 'mean'],
    'Date_': lambda x: (ref_date - x.max()).days
}
for col in basket_type_cols:
    agg_rules[col] = 'sum'

customer_agg = basket_summary.groupby('LoyaltyCard_ID').agg(agg_rules)
customer_agg.columns = ['Total_Spend', 'Total_Visits', 'Avg_Basket_Value', 'Recency'] + basket_type_cols
customer_agg.reset_index(inplace=True)

# Merge Status
df3['Status'] = df3['Status'].astype(str).str.lower().replace(['na', 'nan'], 'Unknown')
customer_agg = pd.merge(customer_agg, df3[['Cardholder', 'Status']], left_on='LoyaltyCard_ID', right_on='Cardholder', how='left')
customer_agg['Status'].fillna('Unknown', inplace=True)

# Filter Outliers
customer_agg_clean = customer_agg[customer_agg['Total_Visits'] < 500].copy()

# Features & Scaling
features = ['Total_Spend', 'Total_Visits', 'Avg_Basket_Value', 'Recency'] + basket_type_cols
status_dummies = pd.get_dummies(customer_agg_clean['Status'], prefix='Status')
X = pd.concat([customer_agg_clean[features], status_dummies], axis=1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# =============================================================================
# 4. ΔΥΝΑΜΙΚΗ ΕΠΙΛΟΓΗ K (Silhouette 3-10)
# =============================================================================
print("\n--- 4. Εύρεση Βέλτιστου K ---")

best_k = 3
best_score = -1
scores = []
k_range = range(3, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    scores.append(score)
    print(f"k={k}, Silhouette Score={score:.4f}")
    
    if score > best_score:
        best_score = score
        best_k = k

print(f"\n--> Βέλτιστο k: {best_k}")

# =============================================================================
# 5. ΤΕΛΙΚΟ CLUSTERING & HEATMAPS
# =============================================================================
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
customer_agg_clean['Customer_Cluster'] = kmeans_final.fit_predict(X_scaled)

# Υπολογισμός Προφίλ
cluster_profile = customer_agg_clean.groupby('Customer_Cluster').mean(numeric_only=True)

# Heatmap 1: Σύνδεση Customer Segments με Basket Types
cluster_basket_counts = cluster_profile[basket_type_cols]
# Μετατροπή σε ποσοστά (ανά γραμμή)
cluster_basket_mix = cluster_basket_counts.div(cluster_basket_counts.sum(axis=1), axis=0)

plt.figure(figsize=(10, 6))
sns.heatmap(cluster_basket_mix, annot=True, cmap='Greens', fmt='.1%')
plt.title(f'Basket Mix per Customer Segment (k={best_k})')
plt.ylabel('Customer Cluster')
plt.xlabel('Basket Type')
plt.show()

# Heatmap 2: Γενικό Προφίλ (Scaled)
cluster_profile_scaled = pd.DataFrame(scaler.fit_transform(cluster_profile), columns=cluster_profile.columns)
plt.figure(figsize=(12, 6))
sns.heatmap(cluster_profile_scaled.T, cmap='RdBu_r', center=0, annot=True, fmt='.2f')
plt.title('Customer Segments Profile (Standardized)')
plt.show()

# =============================================================================
# 6. ΔΥΝΑΜΙΚΗ ΕΡΜΗΝΕΙΑ & ΟΝΟΜΑΤΟΔΟΣΙΑ CLUSTERS (ΔΙΟΡΘΩΜΕΝΟ)
# =============================================================================
print("\n--- 6. Αυτόματη Ερμηνεία Clusters ---")

# --- ΔΙΟΡΘΩΣΗ: Προσθέτουμε τα Status Dummies στο dataframe ανάλυσης ---
status_dummies = pd.get_dummies(customer_agg_clean['Status'], prefix='Status')
# Αφαιρούμε τις στήλες αν υπάρχουν ήδη για να μην έχουμε διπλότυπα
cols_to_drop = [c for c in status_dummies.columns if c in customer_agg_clean.columns]
customer_agg_clean = customer_agg_clean.drop(columns=cols_to_drop)
# Ενώνουμε
customer_agg_clean = pd.concat([customer_agg_clean, status_dummies], axis=1)

# Επαναϋπολογισμός του προφίλ τώρα που έχουμε όλα τα δεδομένα
cluster_profile = customer_agg_clean.groupby('Customer_Cluster').mean(numeric_only=True)

# 1. Βρίσκουμε το κυρίαρχο Basket Type
# Παίρνουμε μόνο τις στήλες που ξεκινάνε από 'Basket_Type_'
basket_cols_dynamic = [c for c in cluster_profile.columns if 'Basket_Type_' in c]
cluster_basket_mix = cluster_profile[basket_cols_dynamic].div(cluster_profile[basket_cols_dynamic].sum(axis=1), axis=0)

dominant_basket = cluster_basket_mix.idxmax(axis=1) # π.χ. 'Basket_Type_2'
dominant_basket_val = cluster_basket_mix.max(axis=1) # π.χ. 0.65

# 2. Βρίσκουμε το κυρίαρχο Status
status_cols_dynamic = [c for c in cluster_profile.columns if 'Status_' in c]
cluster_status_counts = cluster_profile[status_cols_dynamic] # Εδώ είναι ήδη ποσοστά (0 ή 1 στα dummies -> mean = ποσοστό)

dominant_status = cluster_status_counts.idxmax(axis=1).str.replace('Status_', '')
dominant_status_val = cluster_status_counts.max(axis=1)

# 3. Ορίζουμε τα όρια (Benchmarks)
avg_visits = customer_agg_clean['Total_Visits'].mean()
avg_spend = customer_agg_clean['Total_Spend'].mean()
avg_recency = customer_agg_clean['Recency'].mean()

# ΛΕΞΙΚΟ ΟΝΟΜΑΣΙΑΣ ΚΑΛΑΘΙΩΝ (ΠΡΟΣΑΡΜΟΣΕ ΤΟ ΑΝ ΧΡΕΙΑΖΕΤΑΙ)
# Εδώ πρέπει να αντιστοιχίσεις τα Basket_Type_0, 1, 2 κτλ με αυτό που είδες στο Basket Heatmap
# Αν δεν είσαι σίγουρος, ο κώδικας θα γράψει απλά "Basket Type X Shoppers"
basket_names_map = {
    'Basket_Type_0': 'Snacking (Type 0)',
    'Basket_Type_1': 'Variety (Type 1)',
    'Basket_Type_2': 'Bakery (Type 2)',
    'Basket_Type_3': 'Cheese/Deli (Type 3)',
    'Basket_Type_4': 'Milk (Type 4)'
}

print(f"{'Cluster':<8} | {'Label (Generated Name)':<45} | {'Explanation'}")
print("-" * 120)

cluster_labels = {}

for i in cluster_profile.index:
    # Λήψη δεδομένων
    visits = cluster_profile.loc[i, 'Total_Visits']
    spend = cluster_profile.loc[i, 'Total_Spend']
    recency = cluster_profile.loc[i, 'Recency']
    
    # Basket Info
    dom_bask = dominant_basket[i]
    dom_bask_name = basket_names_map.get(dom_bask, dom_bask) # Παίρνει το όνομα ή το κωδικό αν δεν βρει όνομα
    dom_bask_pct = dominant_basket_val[i]
    
    # Status Info
    dom_stat = dominant_status[i]
    dom_stat_pct = dominant_status_val[i]

    # --- LOGIC ΓΙΑ ΤΗΝ ΟΝΟΜΑΣΙΑ ---
    label = "Mixed / Occasional"
    desc = "Μικτή συμπεριφορά."

    # A. Έλεγχος Αδράνειας (Churn)
    if recency > avg_recency * 1.3 and visits < avg_visits:
        label = "Inactive / Lapsed Customers"
        desc = f"⚠️ Μεγάλο Recency ({recency:.0f} μέρες). "

    # B. Έλεγχος VIP (Loyalty)
    elif spend > avg_spend * 2 or visits > avg_visits * 2:
        label = "⭐ VIP / Loyal Heavy Shoppers"
        desc = f"Πολύ πιστοί (Visits: {visits:.1f}, Spend: {spend:.1f}€). "
        # Αν είναι VIP και αγοράζουν κυρίως ένα πράγμα, το προσθέτουμε
        if dom_bask_pct > 0.5:
            desc += f"Αγαπούν το {dom_bask_name}. "

    # C. Έλεγχος Βάσει Καλαθιού (Product Focus)
    elif dom_bask_pct > 0.45:
        label = f"{dom_bask_name} Shoppers"
        desc = f"Αγοράζουν στοχευμένα {dom_bask_name} ({dom_bask_pct*100:.0f}%). "
        # Αν υπάρχει έντονο δημογραφικό, το προσθέτουμε στο όνομα
        if dom_stat != 'Unknown' and dom_stat_pct > 0.5:
            label += f" ({dom_stat.title()})"

    # D. Έλεγχος Βάσει Δημογραφικού (Αν δεν έχουν άλλο έντονο χαρακτηριστικό)
    elif dom_stat != 'Unknown' and dom_stat_pct > 0.6:
        label = f"{dom_stat.title()} Segments"
        desc = f"Κυρίως δημογραφική ομάδα {dom_stat}. "

    cluster_labels[i] = label
    print(f"{i:<8} | {label:<45} | {desc}")

# Δημιουργία DataFrame για εύκολη αντιγραφή
final_summary = pd.DataFrame.from_dict(cluster_labels, orient='index', columns=['Segment Name'])
final_summary.index.name = 'Cluster ID'
print("\n--- Τελικός Πίνακας Ομάδων ---")
display(final_summary)

1. Οι "Βαριοί" Πελάτες (Behavior Based)
Αυτές οι ομάδες ξεχωρίζουν από τα Total Spend και Total Visits.

Οι "Πιστοί VIP" (The Loyal Whales)

Τι θα δεις: Πολύ σκούρο χρώμα (υψηλή τιμή) στο Total_Spend και Total_Visits. Πολύ χαμηλό (αρνητικό/ανοιχτό) στο Recency.

Ερμηνεία: Είναι οι καλύτεροι πελάτες σου. Έρχονται συνέχεια και ξοδεύουν τα περισσότερα.

Δράση: Προγράμματα επιβράβευσης για να μην φύγουν ποτέ.

Οι "Χαμένοι / Αδρανείς" (Churned / Lapsed)

Τι θα δεις: Πολύ υψηλό Recency (έχουν πολλές μέρες να φανούν). Χαμηλά Total_Visits.

Ερμηνεία: Πελάτες που ψώνιζαν παλιά αλλά σταμάτησαν. Έχουμε κίνδυνο να τους χάσουμε οριστικά.

Δράση: Προσφορές "Win-back" (π.χ. κουπόνι έκπτωσης για την επόμενη αγορά).

Οι "Περιστασιακοί" (One-Offs / Occasional)

Τι θα δεις: Χαμηλό Total_Spend, χαμηλό Total_Visits, μέτριο Recency.

Ερμηνεία: Μπαίνουν τυχαία στο μαγαζί, δεν έχουν σταθερή προτίμηση.

2. Οι "Ειδικοί" Πελάτες (Product Based)
Αυτές οι ομάδες ξεχωρίζουν κοιτώντας το Basket Mix Heatmap (το πράσινο). Θα δεις ότι κάθε ομάδα έχει ένα Basket Type πολύ πιο ψηλά από τα άλλα.

Οι "Τύποι του Πρωινού & Φούρνου" (The Bakery/Breakfast Club)

Τι θα δεις: Πολύ υψηλό ποσοστό στο Basket_Type_Bakery (ή όπως ονομάσαμε το cluster με τα Αρτοσκευάσματα).

Ερμηνεία: Χρησιμοποιούν το σούπερ μάρκετ σαν φούρνο. Έρχονται συχνά για ψωμί/τυρόπιτες.

Δημογραφικά: Συχνά είναι εργαζόμενοι της περιοχής ή ηλικιωμένοι (Elder).

Οι "Γαλατάδες" (The Milk Runners)

Τι θα δεις: Κυριαρχεί το Basket_Type_Milk.

Ερμηνεία: Μπαίνουν βιαστικά μόνο για γάλα και φεύγουν. Είναι "αποστολή ανάγκης".

Δράση: Τοποθέτηση συμπληρωματικών προϊόντων (π.χ. μπισκότα) δίπλα στα γάλατα.

Οι "Snackers" (Νέοι / Φοιτητές / Γραφείο)

Τι θα δεις: Κυριαρχεί το Basket_Type_Snack (Μπισκότα, Σοκολάτες).

Ερμηνεία: Αγοράζουν λιχουδιές. Πιθανώς μικρότερης ηλικίας ή εργαζόμενοι γραφείου για το break τους.

Οι "Νοικοκυρές / Family Refillers"

Τι θα δεις: Υψηλό ποσοστό στο Basket_Type_Variety (το καλάθι ποικιλίας/χαρτικών/γιαουρτιών) και πιθανώς Status: Family.

Ερμηνεία: Κάνουν τα ψώνια του σπιτιού. Αγοράζουν απορρυπαντικά, χαρτικά και τρόφιμα ψυγείου.

Οι "Deli Lovers" (Τυροκομικά / Αλλαντικά)

Τι θα δεις: Κυριαρχεί το Basket_Type_Cheese.

Ερμηνεία: Εστιάζουν στον πάγκο κοπής. Πιθανώς αγοράζουν για τοστ ή μαγειρική στο σπίτι.